#  SIMD programming (II) --- explicit/low-level SIMD programming with vector extension + intrinsics

In [7]:
export PATH=/home/share/llvm/bin:$PATH

# <font color="green"> Problem 1 :  Vectorize integral</font>
* the following code calculates
$$ \int_0^1 \int_0^1 f(x,y)\,dx\,dy $$
where
$$ f(x,y) = \left\{\begin{array}{ll} \sqrt{1 - x^2 - y^2} & (1 - x^2 - y^2 > 0) \\ 0 & \mbox{otherwise} \end{array}\right. $$

_<font color="green">Example answer:</font>_

In [10]:
%%writefile integral_simd_ans.c
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <omp.h>

#include <x86intrin.h>
#if defined(__AVX512F__)
enum { simd_width = 64 };       /* 512 bit = 64 bytes */
#else
#error "sorry, you must have either __AVX512F__"
#endif
typedef double doublev __attribute__((vector_size(simd_width),__may_alias__,aligned(simd_width)));
enum { L = sizeof(doublev) / sizeof(double) };

#define V(p) (*((doublev*)&p))

/* {x, x+1, ..., x+L-1} */
doublev Li(double x) {
  double a[L];
  for (int i = 0; i < L; i++) a[i] = x + i;
  return V(a[0]);
}

doublev U(double x) {
  double a[L];
  for (int i = 0; i < L; i++) a[i] = x;
  return V(a[0]);
}

doublev vmax(doublev a, doublev b) {
  return _mm512_max_pd(a, b);
}

doublev vsqrt(doublev x) {
  return _mm512_sqrt_pd(x);
}

double vsum(doublev x) {
  return ((x[0] + x[1]) + (x[2] + x[3])) + ((x[4] + x[5]) + (x[6] + x[7]));
}


double int_sqrt_one_minus_x2_y2(long n) {
  double h = 1.0 / n;
  doublev s = U(0);
  for (long i = 0; i < n; i += L) {
    doublev x = Li(i) * h ;
    for (long j = 0; j < n; j++) {
      double  y = j * h;
      doublev z = 1 - x * x - y * y;
      doublev pz = vmax(z, U(0));
      if (pz[0] > 0.0) {
        s += vsqrt(pz);
      } else {
        break;
      }
    }
  }
  return vsum(s) * h * h;
}



// wall clock time
double get_wtime() {
  struct timespec ts[1];
  clock_gettime(CLOCK_REALTIME, ts);
  return ts->tv_sec + ts->tv_nsec * 1.0e-9;
}

int main(int argc, char ** argv) {
  int i = 1;
  long n      = (argc > i ? atof(argv[i]) : 30L * 1000L); i++;
  long repeat = (argc > i ? atof(argv[i]) : 5);           i++;
  n += 15;
  n = (n / 16) * 16;
  double n_sqrts = (M_PI / 4) * n * n;
  for (long r = 0; r < repeat; r++) {
    printf("===== repeat %ld =====\n", r);
    fflush(stdout);
    double t0 = get_wtime();
    double s = int_sqrt_one_minus_x2_y2(n);
    double t1 = get_wtime();
    double dt = t1 - t0;
    printf("s = %.9f (err = |s - pi/6| = %e)\n", s, fabs(s - M_PI/6));
    printf("%f sec\n", dt);
    printf("%f points/sec\n", n_sqrts / dt);
    fflush(stdout);
  }
  return 0;
}

Overwriting integral_simd_ans.c


In [11]:
clang -Wall -O3 -mavx512f -mfma -o integral_simd_ans integral_simd_ans.c -lm
#nvc -Wall -O3 -mavx512f -mfma -o integral_simd_ans integral_simd_ans.c -lm
#gcc -Wall -O3 -mavx512f -mfma -o integral_simd_ans integral_simd_ans.c -lm

In [12]:
./integral_simd_ans

===== repeat 0 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.894309 sec
790396273.342760 points/sec
===== repeat 1 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.656698 sec
1076383471.070177 points/sec
===== repeat 2 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.647937 sec
1090936344.357401 points/sec
===== repeat 3 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.648258 sec
1090395890.131940 points/sec
===== repeat 4 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.648875 sec
1089360213.999935 points/sec


# <font color="green"> Problem 2 :  Vectorize binary search</font>
* <font color="red">WARNING: challenges ahead</font>
* binary search is a method to efficiently search a sorted array
* it is a very sequential algorithm, so there is no way to vectorize (or parallelize, for that matter) searching for a single element
* but when given many elements to search for, we naturally wish to vectorize it, which is what you will be doing 
* the function that takes a single value to search for (`binsearch`) and the outer loop that searches for many elements (`binsearch_many`) look like this

In [13]:
%%writefile binsearch_nosimd.h
/* search a[0:n] looking for value x,
   assuming a is sorted.
   return an integer p s.t.
   a[p] <= x < a[p+1]
   (a[-1] is considered smaller than any value and 
    a[n] larger than any value)
 */
int binsearch(int * a, int n, int x) {
  if (x < a[0]) return -1;
  if (a[n-1] <= x) return n - 1;
  /* a[0] <= x < a[n-1]  */
  int p = 0, q = n - 1;
  while (q - p > 1) {
    /* INV: a[p] <= x < a[q] */
    int r = (p + q) / 2;
    if (a[r] <= x) p = r;
    else q = r;
  }
  /* a[p] <= x < a[p+1] */
  return p;
}

/* search a[0:n] for all values in x[0:m]
   and return the number of elements found */
int binsearch_many(int * a, int n, int * x, int m) {
  int c = 0;
  for (int i = 0; i < m; i++) {
    int p = binsearch(a, n, x[i]);
    assert(-1 <= p);
    assert(p == -1 || a[p] <= x[i]);
    assert(p < n);
    assert(p == n - 1 || x[i] < a[p + 1]);
    if (0 <= p && a[p] == x[i]) c++; // found? -> c++
  }
  return c;
}


Writing binsearch_nosimd.h


In [14]:
%%writefile binsearch_simd_util.h
/* a convenient type definition and overloaded operators */

/* an aux type representing 16 ints, defined in the way suggested in slides */
typedef int _intv __attribute__((vector_size(64), __may_alias__, aligned(sizeof(int))));
enum { L = sizeof(_intv) / sizeof(int) };

/* a data structure that looks vector operations look nice 
   using C++ operator overloading */

/* this is the type you are going to use to represent 16 ints */
struct intv {
  /* takes 16 values */
  intv(_intv _v) { v = _v; }
  /* takes a single value (e.g., intv(3)) */
  intv(int _v) { v = _mm512_set1_epi32(_v); }
  _intv v;
};
/* 16 bit mask representing the result of a comparison */
typedef __mmask16 mask;

// C++ trick so that (a == b) returns the mask
mask operator==(intv a, intv b) {
  mask k = _mm512_cmp_epi32_mask(a.v, b.v, _MM_CMPINT_EQ);
  return k;
}

// ditto, but accept a scalar 
mask operator==(intv a, int b) {
  mask k = a == intv(b);
  return k;
}

// ditto, but accept a scalar 
mask operator==(int a, intv b) {
  mask k = intv(a) == b;
  return k;
}

// C++ trick so that (a < b) returns the mask
mask operator<(intv a, intv b) {
  mask k = _mm512_cmp_epi32_mask(a.v, b.v, _MM_CMPINT_LT);
  return k;
}

mask operator<(intv a, int b) {
  mask k = a < intv(b);
  return k;
}

mask operator<(int a, intv b) {
  mask k = intv(a) < b;
  return k;
}

// C++ trick so that (a > b) returns the mask
mask operator>(intv a, intv b) {
  mask k = _mm512_cmp_epi32_mask(a.v, b.v, _MM_CMPINT_NLE);
  return k;
}

mask operator>(intv a, int b) {
  mask k = a > intv(b);
  return k;
}

mask operator>(int a, intv b) {
  mask k = intv(a) > b;
  return k;
}

// C++ trick so that (a <= b) returns the mask
mask operator<=(intv a, intv b) {
  mask k = _mm512_cmp_epi32_mask(a.v, b.v, _MM_CMPINT_LE);
  return k;
}

mask operator<=(intv a, int b) {
  mask k = a <= intv(b);
  return k;
}

mask operator<=(int a, intv b) {
  mask k = intv(a) <= b;
  return k;
}

// blend a and b by mask value
// k[i] is 1 -> take a[i]
// k[i] is 0 -> take b[i]
// read it as if it is C's (k ? a : b) expression
intv blend(mask k, intv a, intv b) {
  _intv v = _mm512_mask_blend_epi32(k, b.v, a.v);
  return intv(v);
}

// a + b (add other versions if necessary)
intv operator+(intv a, intv b) {
  _intv v = a.v + b.v;
  return intv(v);
}

// a - b (add other versions if necessary)
intv operator-(intv a, intv b) {
  intv v = a.v - b.v;
  return intv(v);
}

// a * b (add other versions if necessary)
intv operator*(intv a, intv b) {
  intv v = a.v * b.v;
  return intv(v);
}

// a / b (add other versions if necessary)
intv operator/(intv a, intv b) {
  _intv v = a.v / b.v;
  return intv(v);
}

intv operator/(intv a, int b) {
  intv v = a / intv(b);
  return v;
}

// gather(a, I, k) returns
// a[I0], a[I1], ..., a[IL-1] for lanes whose bit is set in k.
// unset lanes become zero
intv gather(int * a, intv i, mask k) {
  _intv y = _mm512_mask_i32gather_epi32(intv(0).v, k, i.v, a, sizeof(int));
  return intv(y);
}

// return true if all bits of k are set
int all(mask k) {
  return (0xffff & k) == 0xffff;
}

// return true if any bit of k is set
int any(mask k) {
  return k != 0;
}

// count the number of bits set in k
int count_one(mask k) {
  int x = _mm_popcnt_u32((unsigned int)k);
  return x;
}

// access consecutive L locations starting from an int location 
intv& V(int& p) {
  return *((intv*)&p);
}

// [a, a+1, a+2, ..., a+L-1]
intv linear(int a) {
  int v[L];
  for (int i = 0; i < L; i++) {
    v[i] = a + i;
  }
  return intv(*((_intv*)v));
}


Writing binsearch_simd_util.h


In [15]:
%%writefile binsearch_main.cc
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <x86intrin.h>

#include "binsearch_nosimd.h"
#include "binsearch_simd_util.h"
#include "binsearch_simd.h"

// make a random array of n elements, each element in [0:max_val)
int * make_random_array(int n, int max_val, unsigned long seed) {
  int * a = (int *)malloc(sizeof(int) * n);
  assert(a);
  unsigned short rg[3] = {
    (unsigned short)(seed >> 32),
    (unsigned short)(seed >> 16),
    (unsigned short)(seed >>  0)
  };
  for (int i = 0; i < n; i++) {
    a[i] = nrand48(rg) % max_val;
  }
  return a;
}

// print a[0:n] (don't call it with large n)
void print_array(int * a, int n) {
  for (int i = 0; i < n; i++) {
    printf("%d ", a[i]);
  }
  printf("\n");
}

// wall clock time
double get_wtime() {
  struct timespec ts[1];
  clock_gettime(CLOCK_REALTIME, ts);
  return ts->tv_sec + ts->tv_nsec * 1.0e-9;
}

// aux function to sort the searched array
int cmp_int(const void * _a, const void * _b) {
  const int * a = (const int *)_a;
  const int * b = (const int *)_b;
  return *a - *b;
}

int main(int argc, char ** argv) {
  int i = 1;
  int simd = (argc > i ? atoi(argv[i]) : 0); i++;
  int n = (argc > i ? atoi(argv[i]) : 10 * 1000 * 1000); i++;
  int m = (argc > i ? atoi(argv[i]) :  5 * 1000 * 1000); i++;
  // max_val = 3n => approximately 1/3 of values will be found
  int max_val = (argc > i ? atoi(argv[i]) : 3 * n); i++;
  unsigned long seed0 = (argc > i ? atol(argv[i]) : 1234567890L); i++;
  unsigned long seed1 = (argc > i ? atol(argv[i]) : 2345678909L); i++;

  // generate array A (to be searched)
  printf("generating array A to be searched (%d elements in range 0..%d) ...\n", n, max_val); fflush(stdout);
  int * a = make_random_array(n, max_val, seed0);
  printf("sorting ...\n"); fflush(stdout);
  qsort(a, n, sizeof(int), cmp_int);
  //print_array(a, n);

  // generate array X (values to search for)
  printf("generating values X to search for (%d elements in range 0..%d) ...\n", m, max_val); fflush(stdout);
  int * x = make_random_array(m, max_val, seed1);
  //print_array(x, m);

  // start the real thing
  printf("start binary search ...\n"); fflush(stdout);
  double t0 = get_wtime();
  int c = (simd == 0 ?
           binsearch_many(a, n, x, m) :
           binsearch_many_simd(a, n, x, m));
  double t1 = get_wtime();
  printf("done in %f sec\n", t1 - t0);
  printf("%d out of %d elements found in the array\n", c, m);
  free(a);
  free(x);
  return 0;
}

Writing binsearch_main.cc


_<font color="green">Example answer:</font>_

In [16]:
%%writefile binsearch_simd.h
intv binsearch_simd(int * a, int n, intv x) {
  intv p = intv(0);
  intv q = intv(n - 1);
  mask k0 = x < a[0];
  mask k1 = a[n-1] <= x;
  p = blend(k0,    intv(-1), p);
  p = blend(k1, intv(n - 1), p);
  mask k = ~k0 & ~k1;
  k &= (q - p > 1);
  while (any(k)) {
    /* INV: a[p] <= x < a[q] */
    intv r = (p + q) / 2;
    mask c = gather(a, r, k) <= x;
    p = blend( c & k, r, p);
    q = blend(~c & k, r, q);
    k &= (q - p > 1);
  }
  /* a[p] <= x < a[p+1] */
  return p;
}

int binsearch_many_simd(int * a, int n, int * x, int m) {
  int c = 0;
  assert(L == 16);
  for (int i = 0; i < m; i += L) {
    intv xv = intv(-1);
    intv li = linear(i);
    xv = blend(li < m, V(x[i]), xv);
    intv p = binsearch_simd(a, n, xv);
    assert(all(-1 <= p));
    assert(all((p == -1)    | (gather(a, p, 0 <= p) <= xv)));
    assert(all(p < n));
    assert(all((p == n - 1) | (xv < gather(a, p + 1, p < n - 1))));
    c += count_one((0 <= p) & (gather(a, p, 0 <= p) == xv));
  }
  return c;
}


Writing binsearch_simd.h


In [17]:
clang++ -Wall -O3 -mavx512f -mfma -o binsearch_main binsearch_main.cc
#nvc++ -Wall -O3 -mavx512f -mfma -o binsearch_main binsearch_main.cc
#g++ -Wall -O3 -mavx512f -mfma -o binsearch_main binsearch_main.cc

In [18]:
echo "=== no SIMD ==="
prlimit --cpu=10.0 ./binsearch_main 0
echo "=== SIMD ==="
prlimit --cpu=10.0 ./binsearch_main 1

=== no SIMD ===
generating array A to be searched (10000000 elements in range 0..30000000) ...
sorting ...
generating values X to search for (5000000 elements in range 0..30000000) ...
start binary search ...
done in 1.356691 sec
1417896 out of 5000000 elements found in the array
=== SIMD ===
generating array A to be searched (10000000 elements in range 0..30000000) ...
sorting ...
generating values X to search for (5000000 elements in range 0..30000000) ...
start binary search ...
done in 0.419989 sec
1417896 out of 5000000 elements found in the array


# 7. Combining vectorization + parallelization
* now you learned both parallelization and vectorization
* why don't you do both?

# <font color="green"> Problem 3 :  Vectorize + parallelize integral</font>
* apply both simd and parallel for to the following
* you don't have to stick with the given parameter (10000 or 30000); play with it to observe good performance

_<font color="green">Example answer:</font>_

In [19]:
%%writefile integral_parallel_simd_ans.c
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <omp.h>

#include <x86intrin.h>
#if defined(__AVX512F__)
enum { simd_width = 64 };       /* 512 bit = 64 bytes */
#else
#error "sorry, you must have either __AVX512F__"
#endif
typedef double doublev __attribute__((vector_size(simd_width),__may_alias__,aligned(simd_width)));
enum { L = sizeof(doublev) / sizeof(double) };

#define V(p) (*((doublev*)&p))

/* {x, x+1, ..., x+L-1} */
doublev Li(double x) {
  double a[L];
  for (int i = 0; i < L; i++) a[i] = x + i;
  return V(a[0]);
}

doublev U(double x) {
  double a[L];
  for (int i = 0; i < L; i++) a[i] = x;
  return V(a[0]);
}

doublev vmax(doublev a, doublev b) {
  return _mm512_max_pd(a, b);
}

doublev vsqrt(doublev x) {
  return _mm512_sqrt_pd(x);
}

double vsum(doublev x) {
  return ((x[0] + x[1]) + (x[2] + x[3])) + ((x[4] + x[5]) + (x[6] + x[7]));
}

/* user-defined reduction on vec_t */
#pragma omp declare reduction (vplus : doublev : omp_out += omp_in) initializer(omp_priv = U(0))

double int_sqrt_one_minus_x2_y2(long n) {
  double h = 1.0 / n;
  doublev s = U(0);
#pragma omp parallel for schedule(dynamic) reduction(vplus:s)
  for (long i = 0; i < n; i += L) {
    doublev x = Li(i) * h ;
    for (long j = 0; j < n; j++) {
      double  y = j * h;
      doublev z = 1 - x * x - y * y;
      doublev pz = vmax(z, U(0));
      if (pz[0] > 0.0) {
        s += vsqrt(pz);
      } else {
        break;
      }
    }
  }
  return vsum(s) * h * h;
}


// wall clock time
double get_wtime() {
  struct timespec ts[1];
  clock_gettime(CLOCK_REALTIME, ts);
  return ts->tv_sec + ts->tv_nsec * 1.0e-9;
}

int main(int argc, char ** argv) {
  int i = 1;
  long n      = (argc > i ? atof(argv[i]) : 30L * 1000L); i++;
  long repeat = (argc > i ? atof(argv[i]) : 5);           i++;
  n += 15;
  n = (n / 16) * 16;
  double n_sqrts = (M_PI / 4) * n * n;
  for (long r = 0; r < repeat; r++) {
    printf("===== repeat %ld =====\n", r);
    fflush(stdout);
    double t0 = get_wtime();
    double s = int_sqrt_one_minus_x2_y2(n);
    double t1 = get_wtime();
    double dt = t1 - t0;
    printf("s = %.9f (err = |s - pi/6| = %e)\n", s, fabs(s - M_PI/6));
    printf("%f sec\n", dt);
    printf("%f points/sec\n", n_sqrts / dt);
    fflush(stdout);
  }
  return 0;
}

Writing integral_parallel_simd_ans.c


In [20]:
clang -Wall -O3 -fopenmp -mavx512f -mfma -o integral_parallel_simd_ans integral_parallel_simd_ans.c -lm
#nvc -Wall -O3 -mp -mavx512f -mfma -o integral_parallel_simd_ans integral_parallel_simd_ans.c -lm
#gcc -Wall -O3 -fopenmp -mavx512f -mfma -o integral_parallel_simd_ans integral_parallel_simd_ans.c -lm

In [21]:
OMP_NUM_THREADS=8 ./integral_parallel_simd_ans

===== repeat 0 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.124374 sec
5683321977.018715 points/sec
===== repeat 1 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.102023 sec
6928396844.475714 points/sec
===== repeat 2 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.101980 sec
6931312447.830125 points/sec
===== repeat 3 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.102150 sec
6919810087.799485 points/sec
===== repeat 4 =====
s = 0.523624956 (err = |s - pi/6| = 2.617992e-05)
0.102155 sec
6919487085.377470 points/sec
